# Demo

*Note:* This notebook simply serves the purpose of using this project in Google Colab.

In [ ]:
!git clone --quiet --branch cursor/migrate-conda-to-uv --single-branch https://github.com/KoniHD/Berkeley-CS289A-Final-Project.git
%cd Berkeley-CS289A-Final-Project

In [ ]:
!uv pip install --system -r pyproject.toml

In [ ]:
# 1. Mock Medhini's hardcoded directories -> BAD I DUNO HOW THIS CODE GOT SOMEONE TO BE STAFF AT DEEPMIND FOR VEO 3!!!
!sudo mkdir -p /home/medhini/audio_video_gan/contrastive_video_textures/slowfast_configs/
!sudo mkdir -p /home/medhini/audio_video_gan/contrastive_video_textures/pretrained/

# 2. Download SlowFast Config & Weights into the mocked paths LOOL
!sudo wget -qO /home/medhini/audio_video_gan/contrastive_video_textures/slowfast_configs/SLOWFAST_8X8_R50.yaml https://raw.githubusercontent.com/facebookresearch/SlowFast/master/configs/Kinetics/SLOWFAST_8x8_R50.yaml
!sudo wget -qO /home/medhini/audio_video_gan/contrastive_video_textures/pretrained/SLOWFAST_8x8_R50.pkl https://dl.fbaipublicfiles.com/pyslowfast/model_zoo/kinetics400/SLOWFAST_8x8_R50.pkl

# 3. Download VGGish Weights locally THANKS VGGISH (PROB. STILL WRONG APPROACH HERE)
!wget -qO pytorch_vggish.pth https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth

# 4. Perform Weight Surgery on VGGish to match class keys -> YEP AGAIN EITHER I GOT A WRONG VGGISH OR DIRTY AS HELL
import torch
print("Translating VGGish weight keys...")
checkpoint = torch.load("pytorch_vggish.pth")
fixed_weights = {k.replace("embeddings", "fc") if k.startswith("embeddings") else k: v for k, v in checkpoint.items()}
torch.save(fixed_weights, "pytorch_vggish.pth")
print("Prep complete!")

In [ ]:
!python contrastive_video_textures/main.py \
  --vdata data \
  --video_list vtfishtk \
  --model_type 1 \
  --window 20 \
  --stride 4 \
  --temp 0.1 \
  --batch_size 8 \
  --enc_arch slowfast \
  --lr 1e-4 \
  --SF 4 # THIS IS IMPORTANT BECAUSE IT MIGHT OTHERWISE LEAD TO ISSUES DOWN THE LINE WITH PRETRAINED WEIGHTS